# PROJECT BEHOLDER - DATASET ANALYZER


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

class IDSDatasetAnalyzer:
    """
    Classe para análise completa de datasets de tráfego de rede
    gerados pelo CICFlowMeter para sistemas IDS.
    """

    def __init__(self, csv_path, output_dir="resultados_analise"):
        """
        Inicializa o analisador.

        Args:
            csv_path (str): Caminho para o arquivo CSV do CICFlowMeter
            output_dir (str): Diretório para salvar os resultados
        """
        self.csv_path = csv_path
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

        # Criar subdiretórios
        (self.output_dir / "graficos").mkdir(exist_ok=True)
        (self.output_dir / "tabelas").mkdir(exist_ok=True)

        print(f"[INFO] Carregando dataset de: {csv_path}")
        self.df = pd.read_csv(csv_path, low_memory=False)
        print(f"[INFO] Dataset carregado: {len(self.df)} fluxos")

        # Converter timestamp para datetime
        if 'timestamp' in self.df.columns:
            self.df['timestamp'] = pd.to_datetime(self.df['timestamp'])

        self.labeled = False

    def rotular_dataset(self, ips_normais=None, ips_atacantes=None,
                        ips_alvos=None, auto_detect=True):
        """
        Rotula o dataset identificando tráfego normal e anômalo.

        Args:
            ips_normais (list): Lista de IPs que geram tráfego normal
            ips_atacantes (list): Lista de IPs que executam ataques
            ips_alvos (list): Lista de IPs alvo dos ataques (contêineres)
            auto_detect (bool): Detectar automaticamente baseado em padrões conhecidos
        """
        print("\n[INFO] Iniciando rotulação do dataset...")

        # Inicializar coluna de labels
        self.df['label'] = 'Normal'
        self.df['attack_type'] = 'Normal'

        if auto_detect:
            # Detectar Slow Loris (duração ~180s, muitos pacotes fwd, poucos/zero bwd)
            slowloris_mask = (
                (self.df['flow_duration'] > 175) &
                (self.df['flow_duration'] < 185) &
                (self.df['tot_fwd_pkts'] > 100) &
                (self.df['tot_bwd_pkts'] < 10)
            )
            self.df.loc[slowloris_mask, 'label'] = 'Anomalo'
            self.df.loc[slowloris_mask, 'attack_type'] = 'Slow_Loris'

            # Detectar NMAP (ICMP com múltiplos pacotes ou scan de portas TCP)
            nmap_icmp_mask = (
                (self.df['protocol'].isin([1, 'ICMP', 'icmp'])) &
                (self.df['tot_fwd_pkts'] > 10)
            )
            self.df.loc[nmap_icmp_mask, 'label'] = 'Anomalo'
            self.df.loc[nmap_icmp_mask, 'attack_type'] = 'NMAP_Scan'

        # Rotulação baseada em IPs específicos
        if ips_atacantes:
            atk_mask = self.df['src_ip'].isin(ips_atacantes)
            self.df.loc[atk_mask, 'label'] = 'Anomalo'

        if ips_alvos and ips_atacantes:
            # Tráfego de atacante para alvo
            target_mask = (
                self.df['src_ip'].isin(ips_atacantes) &
                self.df['dst_ip'].isin(ips_alvos)
            )
            self.df.loc[target_mask, 'label'] = 'Anomalo'

        self.labeled = True

        # Estatísticas de rotulação
        normal_count = (self.df['label'] == 'Normal').sum()
        anomalo_count = (self.df['label'] == 'Anomalo').sum()

        print(f"[INFO] Rotulação concluída:")
        print(f"  - Tráfego Normal: {normal_count} ({normal_count/len(self.df)*100:.2f}%)")
        print(f"  - Tráfego Anômalo: {anomalo_count} ({anomalo_count/len(self.df)*100:.2f}%)")
        print(f"\n[INFO] Distribuição de tipos de ataque:")
        print(self.df['attack_type'].value_counts())

    def gerar_metricas_gerais(self):
        """
        Gera métricas de caracterização geral do dataset.
        """
        print("\n" + "="*70)
        print("1. MÉTRICAS DE CARACTERIZAÇÃO GERAL DO DATASET")
        print("="*70)

        metricas = {}

        # Volume e distribuição temporal
        metricas['total_fluxos'] = len(self.df)
        metricas['duracao_experimento_horas'] = (
            (self.df['timestamp'].max() - self.df['timestamp'].min()).total_seconds() / 3600
        ) if 'timestamp' in self.df.columns else 0

        if self.labeled:
            metricas['fluxos_normais'] = (self.df['label'] == 'Normal').sum()
            metricas['fluxos_anomalos'] = (self.df['label'] == 'Anomalo').sum()
            metricas['percentual_normal'] = metricas['fluxos_normais'] / metricas['total_fluxos'] * 100
            metricas['percentual_anomalo'] = metricas['fluxos_anomalos'] / metricas['total_fluxos'] * 100

        # Diversidade de protocolos e portas
        metricas['protocolos_unicos'] = self.df['protocol'].nunique()
        metricas['ips_origem_unicos'] = self.df['src_ip'].nunique()
        metricas['ips_destino_unicos'] = self.df['dst_ip'].nunique()
        metricas['portas_origem_unicas'] = self.df['src_port'].nunique()
        metricas['portas_destino_unicas'] = self.df['dst_port'].nunique()

        # Taxa de fluxos
        if metricas['duracao_experimento_horas'] > 0:
            metricas['fluxos_por_hora'] = metricas['total_fluxos'] / metricas['duracao_experimento_horas']
            metricas['fluxos_por_minuto'] = metricas['fluxos_por_hora'] / 60

        # Imprimir métricas
        print(f"\nVolume de Dados:")
        print(f"  Total de Fluxos Capturados: {metricas['total_fluxos']:,}")
        print(f"  Duração do Experimento: {metricas['duracao_experimento_horas']:.2f} horas")

        if self.labeled:
            print(f"\nDistribuição de Classes:")
            print(f"  Fluxos Normais: {metricas['fluxos_normais']:,} ({metricas['percentual_normal']:.2f}%)")
            print(f"  Fluxos Anômalos: {metricas['fluxos_anomalos']:,} ({metricas['percentual_anomalo']:.2f}%)")

        print(f"\nDiversidade:")
        print(f"  Protocolos Únicos: {metricas['protocolos_unicos']}")
        print(f"  IPs de Origem Únicos: {metricas['ips_origem_unicos']}")
        print(f"  IPs de Destino Únicos: {metricas['ips_destino_unicos']}")
        print(f"  Portas de Origem Únicas: {metricas['portas_origem_unicas']}")
        print(f"  Portas de Destino Únicas: {metricas['portas_destino_unicas']}")

        if metricas['duracao_experimento_horas'] > 0:
            print(f"\nTaxas de Fluxo:")
            print(f"  Fluxos/Hora: {metricas['fluxos_por_hora']:.2f}")
            print(f"  Fluxos/Minuto: {metricas['fluxos_por_minuto']:.2f}")

        return metricas

    def gerar_tabelas_resumo(self):
        """
        Gera tabelas resumo em formato LaTeX e CSV.
        """
        print("\n" + "="*70)
        print("2. GERANDO TABELAS RESUMO")
        print("="*70)

        # Tabela 1: Resumo Geral do Experimento
        duracao = (self.df['timestamp'].max() - self.df['timestamp'].min()).total_seconds() / 3600

        tabela1 = pd.DataFrame({
            'Métrica': [
                'Duração Total (h)',
                'Total de Fluxos',
                'Fluxos Normais',
                'Fluxos Anômalos',
                'Protocolos Distintos',
                'IPs Únicos (Origem)',
                'IPs Únicos (Destino)'
            ],
            'Valor': [
                f"{duracao:.2f}",
                len(self.df),
                f"{(self.df['label'] == 'Normal').sum()} ({(self.df['label'] == 'Normal').sum()/len(self.df)*100:.1f}%)" if self.labeled else 'N/A',
                f"{(self.df['label'] == 'Anomalo').sum()} ({(self.df['label'] == 'Anomalo').sum()/len(self.df)*100:.1f}%)" if self.labeled else 'N/A',
                self.df['protocol'].nunique(),
                self.df['src_ip'].nunique(),
                self.df['dst_ip'].nunique()
            ]
        })

        print("\n--- Tabela 1: Resumo Geral do Experimento ---")
        print(tabela1.to_string(index=False))
        tabela1.to_csv(self.output_dir / "tabelas" / "tabela1_resumo_geral.csv", index=False)
        tabela1.to_latex(self.output_dir / "tabelas" / "tabela1_resumo_geral.tex", index=False)

        # Tabela 2: Distribuição por Tipo de Ataque (se rotulado)
        if self.labeled:
            attack_stats = self.df.groupby('attack_type').agg({
                'flow_duration': ['count', 'mean'],
            }).round(2)
            attack_stats.columns = ['Quantidade', 'Duração Média (s)']
            attack_stats['% do Total'] = (attack_stats['Quantidade'] / len(self.df) * 100).round(2)
            attack_stats = attack_stats.reset_index()
            attack_stats.columns = ['Tipo de Tráfego', 'Quantidade', 'Duração Média (s)', '% do Total']

            print("\n--- Tabela 2: Distribuição por Tipo de Ataque ---")
            print(attack_stats.to_string(index=False))
            attack_stats.to_csv(self.output_dir / "tabelas" / "tabela2_distribuicao_ataques.csv", index=False)
            attack_stats.to_latex(self.output_dir / "tabelas" / "tabela2_distribuicao_ataques.tex", index=False)

        # Tabela 3: Estatísticas de Features Principais
        features_interesse = [
            'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts',
            'totlen_fwd_pkts', 'totlen_bwd_pkts', 'flow_byts_s', 'flow_pkts_s'
        ]

        stats_list = []
        for feature in features_interesse:
            if feature in self.df.columns:
                stats_list.append({
                    'Feature': feature,
                    'Média': f"{self.df[feature].mean():.2f}",
                    'Mediana': f"{self.df[feature].median():.2f}",
                    'Desvio Padrão': f"{self.df[feature].std():.2f}",
                    'Mínimo': f"{self.df[feature].min():.2f}",
                    'Máximo': f"{self.df[feature].max():.2f}"
                })

        tabela3 = pd.DataFrame(stats_list)
        print("\n--- Tabela 3: Estatísticas de Features Principais ---")
        print(tabela3.to_string(index=False))
        tabela3.to_csv(self.output_dir / "tabelas" / "tabela3_estatisticas_features.csv", index=False)
        tabela3.to_latex(self.output_dir / "tabelas" / "tabela3_estatisticas_features.tex", index=False)

        # Tabela 4: Distribuição de Protocolos
        protocol_dist = self.df['protocol'].value_counts()
        protocol_pct = (protocol_dist / len(self.df) * 100).round(2)
        tabela4 = pd.DataFrame({
            'Protocolo': protocol_dist.index,
            'Quantidade': protocol_dist.values,
            '% do Total': protocol_pct.values
        })

        print("\n--- Tabela 4: Distribuição de Protocolos ---")
        print(tabela4.to_string(index=False))
        tabela4.to_csv(self.output_dir / "tabelas" / "tabela4_distribuicao_protocolos.csv", index=False)
        tabela4.to_latex(self.output_dir / "tabelas" / "tabela4_distribuicao_protocolos.tex", index=False)

        # Tabela 5: Top 10 Portas Mais Utilizadas
        top_dst_ports = self.df['dst_port'].value_counts().head(10)
        tabela5 = pd.DataFrame({
            'Porta Destino': top_dst_ports.index,
            'Quantidade de Fluxos': top_dst_ports.values,
            '% do Total': (top_dst_ports.values / len(self.df) * 100).round(2)
        })

        print("\n--- Tabela 5: Top 10 Portas de Destino ---")
        print(tabela5.to_string(index=False))
        tabela5.to_csv(self.output_dir / "tabelas" / "tabela5_top_portas.csv", index=False)
        tabela5.to_latex(self.output_dir / "tabelas" / "tabela5_top_portas.tex", index=False)

    def gerar_visualizacoes(self):
        """
        Gera todas as visualizações e gráficos.
        """
        print("\n" + "="*70)
        print("3. GERANDO VISUALIZAÇÕES")
        print("="*70)

        # 1. Distribuição de Classes (Normal vs Anômalo) - APENAS PIZZA
        if self.labeled:
            # Tamanho reduzido pois agora é apenas um gráfico
            fig, ax = plt.subplots(figsize=(8, 6))

            # Gráfico de pizza
            labels_count = self.df['label'].value_counts()
            ax.pie(labels_count.values, labels=labels_count.index, autopct='%1.1f%%',
                      startangle=90, colors=['#2ecc71', '#e74c3c'])
            ax.set_title('Distribuição de Classes\n(Normal vs Anômalo)', fontsize=12, fontweight='bold')

            # Garante que o gráfico de pizza seja desenhado como um círculo
            ax.axis('equal')

            plt.tight_layout()
            plt.savefig(self.output_dir / "graficos" / "01_distribuicao_classes.png", dpi=300, bbox_inches='tight')
            plt.close()
            print("  ✓ Gráfico 1: Distribuição de Classes (Setores)")

        # 2. Distribuição de Protocolos
        fig, ax = plt.subplots(figsize=(10, 6))
        protocol_counts = self.df['protocol'].value_counts().head(10)
        ax.barh(range(len(protocol_counts)), protocol_counts.values, color='coral')
        ax.set_yticks(range(len(protocol_counts)))
        ax.set_yticklabels(protocol_counts.index)
        ax.set_xlabel('Quantidade de Fluxos')
        ax.set_title('Distribuição dos 10 Protocolos Mais Utilizados', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(self.output_dir / "graficos" / "02_distribuicao_protocolos.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Gráfico 2: Distribuição de Protocolos")

        # 3. Série Temporal de Tráfego
        if 'timestamp' in self.df.columns:
            fig, ax = plt.subplots(figsize=(14, 5))
            self.df.set_index('timestamp').resample('5T').size().plot(ax=ax, color='steelblue', linewidth=2)
            ax.set_ylabel('Fluxos por 5 minutos')
            ax.set_xlabel('Tempo')
            ax.set_title('Série Temporal do Tráfego de Rede', fontsize=12, fontweight='bold')
            ax.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(self.output_dir / "graficos" / "03_serie_temporal.png", dpi=300, bbox_inches='tight')
            plt.close()
            print("  ✓ Gráfico 3: Série Temporal")

        # 4. Histogramas de Features Principais
        features = ['flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts', 'flow_byts_s']
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.ravel()

        for idx, feature in enumerate(features):
            if feature in self.df.columns:

                # Remover outliers extremos para melhor visualização
                q1 = self.df[feature].quantile(0.01)
                q99 = self.df[feature].quantile(0.99)
                data_filtered = self.df[(self.df[feature] >= q1) & (self.df[feature] <= q99)][feature]

                axes[idx].hist(data_filtered, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
                axes[idx].set_xlabel(feature)
                axes[idx].set_ylabel('Frequência')
                axes[idx].set_title(f'Distribuição de {feature}', fontweight='bold')
                axes[idx].grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig(self.output_dir / "graficos" / "04_histogramas_features.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Gráfico 4: Histogramas de Features")

        # 5. Boxplots Normal vs Anômalo
        if self.labeled:
            features_box = ['flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts', 'flow_pkts_s']
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            axes = axes.ravel()

            for idx, feature in enumerate(features_box):
                if feature in self.df.columns:
                    data_to_plot = [
                        self.df[self.df['label'] == 'Normal'][feature].dropna(),
                        self.df[self.df['label'] == 'Anomalo'][feature].dropna()
                    ]
                    bp = axes[idx].boxplot(data_to_plot, labels=['Normal', 'Anômalo'],
                                           patch_artist=True, showfliers=False)
                    bp['boxes'][0].set_facecolor('#2ecc71')
                    bp['boxes'][1].set_facecolor('#e74c3c')
                    axes[idx].set_ylabel(feature)
                    axes[idx].set_title(f'Comparação: {feature}', fontweight='bold')
                    axes[idx].grid(axis='y', alpha=0.3)

            plt.tight_layout()
            plt.savefig(self.output_dir / "graficos" / "05_boxplots_normal_vs_anomalo.png", dpi=300, bbox_inches='tight')
            plt.close()
            print("  ✓ Gráfico 5: Boxplots Normal vs Anômalo")

        # 6. Heatmap de Correlação
        features_corr = [
            'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts',
            'totlen_bwd_pkts', 'flow_byts_s', 'flow_pkts_s', 'pkt_len_mean'
        ]
        features_corr = [f for f in features_corr if f in self.df.columns]

        if len(features_corr) > 2:
            fig, ax = plt.subplots(figsize=(12, 10))
            corr_matrix = self.df[features_corr].corr()
            sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                       center=0, square=True, linewidths=1, ax=ax,
                       cbar_kws={"shrink": 0.8})
            ax.set_title('Matriz de Correlação entre Features', fontsize=12, fontweight='bold', pad=20)
            plt.tight_layout()
            plt.savefig(self.output_dir / "graficos" / "06_heatmap_correlacao.png", dpi=300, bbox_inches='tight')
            plt.close()
            print("  ✓ Gráfico 6: Heatmap de Correlação")

        # 7. Top 10 Portas de Destino
        fig, ax = plt.subplots(figsize=(10, 6))
        top_ports = self.df['dst_port'].value_counts().head(10)
        ax.barh(range(len(top_ports)), top_ports.values, color='mediumpurple')
        ax.set_yticks(range(len(top_ports)))
        ax.set_yticklabels(top_ports.index)
        ax.set_xlabel('Quantidade de Fluxos')
        ax.set_title('Top 10 Portas de Destino Mais Acessadas', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(self.output_dir / "graficos" / "07_top_portas.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Gráfico 7: Top 10 Portas")

        print(f"\n[INFO] Todas as visualizações salvas em: {self.output_dir / 'graficos'}")

    def analise_qualidade_dataset(self):
        """
        Analisa a qualidade do dataset (valores ausentes, duplicatas, etc.)
        """
        print("\n" + "="*70)
        print("4. ANÁLISE DE QUALIDADE DO DATASET")
        print("="*70)

        # Valores ausentes
        print("\nValores Ausentes por Coluna:")
        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df) * 100).round(2)
        missing_df = pd.DataFrame({
            'Coluna': missing.index,
            'Valores Ausentes': missing.values,
            '% do Total': missing_pct.values
        })
        missing_df = missing_df[missing_df['Valores Ausentes'] > 0].sort_values('Valores Ausentes', ascending=False)

        if len(missing_df) > 0:
            print(missing_df.to_string(index=False))
        else:
            print("  ✓ Nenhum valor ausente encontrado!")

        # Duplicatas
        duplicates = self.df.duplicated().sum()
        print(f"\nRegistros Duplicados: {duplicates} ({duplicates/len(self.df)*100:.2f}%)")

        # Valores infinitos
        inf_counts = {}
        for col in self.df.select_dtypes(include=[np.number]).columns:
            inf_count = np.isinf(self.df[col]).sum()
            if inf_count > 0:
                inf_counts[col] = inf_count

        if inf_counts:
            print("\nValores Infinitos Encontrados:")
            for col, count in inf_counts.items():
                print(f"  {col}: {count}")
        else:
            print("\n  ✓ Nenhum valor infinito encontrado!")

        # Salvar relatório de qualidade
        with open(self.output_dir / "relatorio_qualidade.txt", "w") as f:
            f.write("RELATÓRIO DE QUALIDADE DO DATASET\n")
            f.write("="*70 + "\n\n")
            f.write(f"Total de Registros: {len(self.df)}\n")
            f.write(f"Registros Duplicados: {duplicates} ({duplicates/len(self.df)*100:.2f}%)\n\n")
            f.write("Valores Ausentes:\n")
            f.write(missing_df.to_string(index=False) if len(missing_df) > 0 else "Nenhum\n")

    def analise_assinatura_ataques(self):
        """
        Analisa características específicas de cada tipo de ataque.
        """
        if not self.labeled:
            print("\n[AVISO] Dataset não rotulado. Execute rotular_dataset() primeiro.")
            return

        print("\n" + "="*70)
        print("5. ANÁLISE DE ASSINATURA DOS ATAQUES")
        print("="*70)

        # Análise por tipo de ataque
        for attack_type in self.df['attack_type'].unique():
            if attack_type == 'Normal':
                continue

            print(f"\n--- {attack_type} ---")
            attack_data = self.df[self.df['attack_type'] == attack_type]

            print(f"Quantidade de Fluxos: {len(attack_data)}")
            print(f"Duração Média: {attack_data['flow_duration'].mean():.2f}s (std: {attack_data['flow_duration'].std():.2f}s)")
            print(f"Pacotes Forward Médio: {attack_data['tot_fwd_pkts'].mean():.2f}")
            print(f"Pacotes Backward Médio: {attack_data['tot_bwd_pkts'].mean():.2f}")
            print(f"Taxa de Fluxos com Bwd=0: {(attack_data['tot_bwd_pkts'] == 0).sum() / len(attack_data) * 100:.2f}%")

            # Portas mais atacadas
            if len(attack_data) > 0:
                top_ports = attack_data['dst_port'].value_counts().head(5)
                print("Portas Alvo Mais Frequentes:")
                for port, count in top_ports.items():
                    print(f"  Porta {port}: {count} fluxos")

    def gerar_relatorio_completo(self):
        """
        Gera um relatório completo em texto.
        """
        print("\n" + "="*70)
        print("GERANDO RELATÓRIO COMPLETO")
        print("="*70)

        relatorio_path = self.output_dir / "relatorio_completo.txt"

        with open(relatorio_path, "w", encoding='utf-8') as f:
            f.write("="*70 + "\n")
            f.write("RELATÓRIO COMPLETO DE ANÁLISE DO DATASET IDS\n")
            f.write("="*70 + "\n")
            f.write(f"Data de Geração: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Dataset Analisado: {self.csv_path}\n")
            f.write(f"Total de Fluxos: {len(self.df):,}\n")
            f.write("="*70 + "\n\n")

            # Métricas gerais
            f.write("1. MÉTRICAS GERAIS\n")
            f.write("-"*70 + "\n")
            duracao = (self.df['timestamp'].max() - self.df['timestamp'].min()).total_seconds() / 3600
            f.write(f"Duração do Experimento: {duracao:.2f} horas\n")
            f.write(f"Protocolos Únicos: {self.df['protocol'].nunique()}\n")
            f.write(f"IPs de Origem Únicos: {self.df['src_ip'].nunique()}\n")
            f.write(f"IPs de Destino Únicos: {self.df['dst_ip'].nunique()}\n\n")

            if self.labeled:
                f.write("2. DISTRIBUIÇÃO DE CLASSES\n")
                f.write("-"*70 + "\n")
                f.write(f"Tráfego Normal: {(self.df['label'] == 'Normal').sum():,} ")
                f.write(f"({(self.df['label'] == 'Normal').sum()/len(self.df)*100:.2f}%)\n")
                f.write(f"Tráfego Anômalo: {(self.df['label'] == 'Anomalo').sum():,} ")
                f.write(f"({(self.df['label'] == 'Anomalo').sum()/len(self.df)*100:.2f}%)\n\n")

                f.write("3. DISTRIBUIÇÃO POR TIPO DE ATAQUE\n")
                f.write("-"*70 + "\n")
                for attack_type, count in self.df['attack_type'].value_counts().items():
                    f.write(f"{attack_type}: {count:,} ({count/len(self.df)*100:.2f}%)\n")

            f.write("\n" + "="*70 + "\n")
            f.write("Análise completa disponível nos diretórios:\n")
            f.write(f"  - Tabelas: {self.output_dir / 'tabelas'}\n")
            f.write(f"  - Gráficos: {self.output_dir / 'graficos'}\n")
            f.write("="*70 + "\n")

        print(f"\n[INFO] Relatório completo salvo em: {relatorio_path}")

    def executar_analise_completa(self):
        """
        Executa todas as análises em sequência.
        """
        print("\n" + "="*80)
        print(" "*20 + "INICIANDO ANÁLISE COMPLETA DO DATASET")
        print("="*80)

        self.gerar_metricas_gerais()
        self.gerar_tabelas_resumo()
        self.gerar_visualizacoes()
        self.analise_qualidade_dataset()

        if self.labeled:
            self.analise_assinatura_ataques()

        self.gerar_relatorio_completo()

        print("\n" + "="*80)
        print(" "*25 + "ANÁLISE CONCLUÍDA COM SUCESSO!")
        print("="*80)
        print(f"\nTodos os resultados foram salvos em: {self.output_dir.absolute()}")
        print("\nArquivos gerados:")
        print(f"  📊 Gráficos: {len(list((self.output_dir / 'graficos').glob('*.png')))} arquivos PNG")
        print(f"  📋 Tabelas: {len(list((self.output_dir / 'tabelas').glob('*.csv')))} arquivos CSV")
        print(f"  📄 Relatórios: {len(list(self.output_dir.glob('*.txt')))} arquivos TXT")

## Iniciando Analyzer

In [ ]:
analyzer = IDSDatasetAnalyzer(
  csv_path="./samples/REAL_24h.csv",
  output_dir="./tmp/REAL_24H"
)

## Colocando Labels do Ambiente

In [ ]:
analyzer.rotular_dataset(
  ips_atacantes=['192.168.1.13'],  # IP da máquina host que executa ataques
  ips_alvos=['192.168.1.10', '192.168.1.11'],  # IPs dos contêineres
  auto_detect=False  # Detecta automaticamente baseado em padrões
)

## Faz Análise Completa

In [ ]:
analyzer.executar_analise_completa()